# Solving models in ModelFlow

A ModelFlow `model` instance is a *callable*: applied to a Pandas DataFrame it solves the model
over a time sample and returns a new DataFrame with the solution.

```python
result = mmodel(df, '2021q1', '2035q4')            # solve df over the sample
result = mmodel(df, '2021q1', '2035q4', silent=0)  # ... with progress output
```

Behind the call, ModelFlow translates the model's equations into Python source code
(optionally compiled with numba), determines a solve order from the dependency graph,
and runs a solver appropriate for the model's structure. Three structural properties decide
almost everything:

* **Leads.** If any equation refers to a *future* value (`X(+1)`), the periods cannot be
  solved one at a time: all periods are simultaneous and must be *stacked* into one big
  system.
* **Simultaneity.** If the current-period dependency graph is acyclic (*topological* model),
  a single sweep through the equations in the right order solves a period exactly.
  Otherwise the simultaneous block must be solved iteratively (Gauss-Seidel or Newton).
* **Equation form.** Normalized or implicit — see the next section. Implicit equations
  *require* a Newton solver.

## Equation types: normalized and implicit

**Normalized** equations have the endogenous variable alone on the left-hand side:

$$y_t = F(y_t, x_t, \dots)$$

```
FRML <> y = 0.5 * y(-1) + 0.2 * x $
```

A normalized equation *assigns* its variable, so a Gauss-Seidel sweep (or a topological
sweep for acyclic models) can solve it directly: evaluate the right-hand side, store the
result in `y`, repeat until convergence.

**Implicit** (un-normalized, residual-form) equations cannot — or should not — be solved
for their unknown analytically. They are written as *residual equations*: the equation
variable is a residual `..___RES`, and the actual unknown is declared with `endo=` in the
FRML name:

$$y^{RES}_t = G(y_t, x_t, \dots) \quad\text{solve for } y_t \text{ such that } y^{RES}_t = 0$$

```
FRML <endo=y> y___RES = y**2 + log(y) - x $
```

No sweep can assign `y` here — only a Newton solver can, by moving the declared unknown
until the residual is zero. The helper `modelmanipulation.un_normalize_expression` (and the
`normal(..., implicit=True)` pipeline in `modelnormalize`) rewrites normalized equations to
this form when a model is deliberately kept implicit; `modelmanipulation.un_normalize_model`
does it for a whole model.

The model records which world it lives in:

| Property | Meaning |
|---|---|
| `mmodel.normalized` | no `___RES` equations — every solver is available |
| `mmodel.implicit` | *all* equations are residual-form |
| `mmodel.hybrid` | a mix of normalized and implicit equations |

**Consequence for solver choice:**

| Equation form | Solvers |
|---|---|
| normalized only | `xgenr`, `sim`, `sim1d` (and their `_ng` versions), legacy `newton` and `newtonstack`, and the feedback-set solvers `newton_fbmin_ng` / `newtonstack_fbmin_ng` |
| fully implicit only | legacy `newton_un_normalized` / `newtonstack_un_normalized` |
| any form — normalized, implicit or hybrid | the *unified* Newton solvers `newton_ng`, `newtonstack_ng` and legacy `newton_implicit` / `newtonstack_implicit` |

The unified solvers use a per-equation mask to select the residual convention (`G(y)` on
residual rows, `F(y) - y` on normalized rows), so the same solver handles all three forms.
A hybrid model therefore needs one of them.

## Automatic solver dispatch

When the model is called without an explicit `solver=`, `model.__call__` inspects the
structure and picks a default:

| Model structure | Dispatched solver |
|---|---|
| leads (`maxlead >= 1`), normalized | `newtonstack` |
| leads, fully implicit | `newtonstack_un_normalized` |
| leads, hybrid | `newtonstack_ng` |
| no leads, normalized, current-period graph acyclic (`istopo`) | `xgenr` |
| no leads, normalized, simultaneous | `sim` |
| no leads, fully implicit | `newton_un_normalized` |
| no leads, hybrid | `newton_ng` |

A hybrid model — a mix of normalized and residual-form equations — needs a unified Newton
solver that handles both residual conventions in one system, so it is dispatched to the `_ng`
solvers (since October 2026; before, hybrids went to the `_un_normalized` solvers, which are
meant for fully implicit models).

Any solver can be forced with the `solver=` option, including the next-generation family by
its `_ng` name:

```python
result = mmodel(df, solver='newton_ng')
result = mmodel(df, solver='newtonstack_ng', split=1)
```

*The plan is to let the dispatch table select the `_ng` solvers once they are fully proven;
for now the legacy solvers remain the default for normalized and fully implicit models.*

### Options are sticky

`model.__call__` remembers the options of the previous call and merges them into the next
one (`{**self.oldkwargs, **kwargs}`). Explicitly passed options always win, but an option
you *stop mentioning* keeps its old value. This is convenient in interactive work — set
`ljit=1` once — but surprising when switching solvers: a `jacobian='fd'` left over from a
`newton_fbmin_ng` call will leak into a later `newtonstack_fbmin_ng` call. Reset with:

```python
result = mmodel(df, reset_options=True, ...)   # forget all remembered options
```
When debugging odd solver behaviour, inspect `mmodel.oldkwargs` first.

Options a solver does not know are ignored without a warning — a misspelled option, or a
model option such as `use_fbmin` passed to a solve, simply has no effect.

## Common options

These options work across (nearly) all solvers. Positional: `mmodel(databank, start, end, ...)`.

| Option | Default | Meaning |
|---|---|---|
| `start`, `end` | current sample | first / last period to solve (also settable via `mmodel.smpl` / `set_smpl`) |
| `solver` | structural guess | select the solver by name (see dispatch table) |
| `silent` | `1` | suppress progress output |
| `stats` | `0` | print timing / floating-point-operation statistics after the solve |
| `timeit` / `timeon` | `0` | fine-grained timers for the phases of the solve (`timeit` in the `_ng` and Newton solvers, `timeon` in legacy `sim`) |
| `max_iterations` | solver-specific | iteration limit (Gauss-Seidel sweeps or Newton steps) |
| `alfa` | `1.0` | damping of the Gauss-Seidel update |
| `conv` | `'*'` | variable pattern the convergence test is applied to (Gauss-Seidel) |
| `absconv` | `0.01` | only variables with `abs(value) >= absconv` enter the convergence test |
| `relconv` | `1e-7` | converged when every tested variable changes relatively less than this |
| `ljit` | `False` | compile the model evaluation with numba (first call pays compilation, later calls are fast; compiled code is cached on disk via a transpiled file in `modelsource/`) |
| `stringjit` | `False` | jit from an in-memory string instead of the transpiled file (no on-disk cache). Use it for benchmarks: legacy and `_ng` solvers can share a transpiled file name, and the legacy loader does not reload a module that is already imported, so a "fresh" legacy compile can silently reuse code another solver compiled |
| `transpile_reset` | `False` | force regeneration of the transpiled solver file |
| `chunk` | `30` | equations per generated function chunk (keeps numba compile time sane). Changes only compile time, never results or run speed: on PAK, 30 compiled fastest; 10 was slightly slower (more functions), 200 clearly slower (numba compile time grows faster than function size) |
| `samedata` | | assume the databank layout is unchanged, reuse the compiled evaluator |
| `init` | `False` | start each period's iteration from the previous period's solution |
| `keep` | `''` | store the solution under this label in `mmodel.keep_solutions` (for comparison plots) |
| `keep_variables` | `''` | restrict what `keep` stores |
| `save` | model default | update `basedf`/`lastdf` bookkeeping |
| `ldumpvar`, `dumpvar`, `dumpwith`, `dumpdecimal` | `False`, `'*'`, ... | record per-iteration values of selected variables in `mmodel.dumpdf` (debugging) |
| `reset_options` | `False` | forget sticky options from earlier calls |
| `keep_residual` | `False` | (`_ng` only) store the last-iteration equation residuals as `mmodel.ng_residual` |
| `progressbar` | `False` | tqdm progress bar (legacy `sim`) |

`use_fbmin` is **not** a solve option — it is set when the model is built; see
the section on `use_fbmin` below. Passed to a solve it is silently ignored.

## `use_fbmin` — ordering the simultaneous core by its minimal feedback set

The simultaneous core of a model can be split into a small **feedback set** (`mmodel.fblist`)
and the rest, which is acyclic once the feedback variables are held fixed
(`mmodel.daglist`, in topological order). `use_fbmin` decides whether the Gauss-Seidel core
is evaluated in that order:

| `use_fbmin` | Core order (`mmodel.coreorder`) |
|---|---|
| `True` (default) | `daglist + fblist`: everything that can be computed exactly from the feedback variables first, the feedback equations last |
| `False` | the normal order of the strongly connected blocks |

It is a **model option**, read once when the model is built (in `superblock()`), so it is set
when creating or loading the model:

```python
mpak, baseline = model.modelload('pak.pcim', run=1, use_fbmin=True)   # the default
mpak, baseline = model.modelload('pak.pcim', run=1, use_fbmin=False)
```

Passing `use_fbmin=` to a solve does nothing (unknown options are ignored without a warning).

What it affects:

* **Gauss-Seidel (`sim`, `sim1d` and their `_ng` versions)** — each sweep evaluates the core in
  this order, so the order changes how many sweeps convergence takes. On PAK the feedback
  ordering is about **1.6 times faster** (warm `sim`: 0.27 s vs 0.44 s without jit,
  0.036 s vs 0.059 s with jit).
* **The converged solution**, within the convergence tolerance: Gauss-Seidel stops on a
  relative-change test, and where it stops depends on the order. On PAK the `sim` solution
  moves by about 5e-5 (relative) between the two settings. Each setting is reproducible
  from load to load.
* **Not** the feedback solvers: `newton_fbmin_ng` and `newtonstack_fbmin_ng` compute
  `fblist` / `daglist` themselves whatever `use_fbmin` is.
* The other `_ng` Newton solvers treat the model as one system and do not use the core
  order. Legacy `newton` iterates on the core, so its path depends on the order.

## The legacy solvers (`modelclass.py`)

### `xgenr` — topological sweep
For *normalized* models whose current-period graph is acyclic. Each period is solved by one
exact pass through the equations in topological order: no iteration, no convergence test,
no Jacobian. This is what the dispatcher picks for definition-style models.

### `sim` — damped Gauss-Seidel
The workhorse for backward-looking simultaneous *normalized* models. Iterates sweeps over
the simultaneous core until the relative-change convergence test passes; the recursive
prolog/epilog parts of the model are evaluated once before/after the core iteration.
Specific options: `alfa` (damping), `first_test` (first iteration to test convergence),
`conv`/`absconv`/`relconv`.

### `sim1d` — Gauss-Seidel on a stuffed 1-D array
Same algorithm as `sim`, but the generated code addresses a one-period 1-D array with
compile-time offsets instead of a 2-D (period x variable) array — better data locality,
usually the fastest Gauss-Seidel variant for large models. Extra option: `cache` for the
numba on-disk cache.

### `res` — one evaluation pass
Not an equation *solver*: evaluates every equation once per period against the input data
(lagged endogenous read from the input, not from computed results) and returns the result.
Used for residual checking and internally to evaluate derivative models.

### `newton` / `newton_un_normalized` — per-period Newton
Newton-Raphson on each period: `newton` on the simultaneous core of a *normalized* model
(the recursive prolog and epilog are evaluated before and after), `newton_un_normalized` for
*fully implicit* models. The Jacobian is built by `modelnewton.newton_diff` and
sparse-LU-factorized at the first period of the solve.

Both default to `newton_reset=1` — a fresh Jacobian on every call — and should keep it: they
hold a single solver that every `nonlin` refresh overwrites with the Jacobian of the period
being solved, so after a solve it belongs to a late period. Reusing it for the next solve
(`newton_reset=0`) can make the first period diverge.

| Option | Default | Meaning |
|---|---|---|
| `nonlin` | `False` | refresh the Jacobian every `nonlin` iterations (for nonlinear models) |
| `newton_reset` | `1` | rebuild the Jacobian at the start of every call |
| `newtonalfa`, `newtonnodamp` | `1.0`, `0` | damping of the Newton step for the first `newtonnodamp` iterations |
| `lnjit` | `False` | jit the *derivative* model |
| `forcenum` | `True` | force numerical (finite-difference) derivatives instead of symbolic |

### `newton_implicit` — unified per-period Newton
Handles normalized, implicit **and hybrid** models: a per-equation mask selects the residual
convention (`G(y)` on `___RES` rows, `F(y) - y` on normalized rows). On a normalized model it
iterates on the simultaneous core with the prolog and epilog evaluated around it; on an
implicit or hybrid model the whole model is one Newton system. Converges on the maximum
residual (`newton_absconv`). Superseded by `newton_ng`, which is far faster on implicit and
hybrid models (see the benchmark).

### `newtonstack` / `newtonstack_un_normalized` / `newtonstack_implicit` — stacked Newton
For models with leads: all periods are stacked into one system of
`n_variables * n_periods` unknowns and solved by Newton with one big sparse LU.
`newtonstack` is for normalized models (the dispatcher's choice for forward-looking models),
`newtonstack_un_normalized` for fully implicit models, and `newtonstack_implicit` is the
unified version for any form — the one `newtonstack_ng` was ported from.

| Option | Default | Meaning |
|---|---|---|
| `nonlin`, `newtonalfa`, `newtonnodamp` | | as for `newton` |
| `newton_reset` | `False` | rebuild the stacked Jacobian at the start of the call |
| `nljit` | `0` | jit the derivative model |
| `nchunk` | `None` (30 when jitted) | chunk size for the derivative model's generated code |
| `forcenum` | `True` | numerical derivatives |

*The stacked LU is factorized with SuperLU using the band-preserving `'NATURAL'` column
ordering by default — on FRB/US MCE this is ~10x faster than fill-reducing orderings,
because the stacked matrix is block-banded by construction.*

## The next-generation solvers (`modelsolver_ng.py`)

The `_ng` family reimplements the solvers on a shared template (`SolverBase`): one common
setup (sample, data preparation, code generation via a single `makelos`), a solver-specific
`prepare`/`iterate`, and a common teardown (result frame, statistics, residual recording).
Nothing shadows the legacy solvers — every entry point carries the `_ng` suffix and is
selected with `solver='<name>_ng'`.

The `_ng` Newton solvers `newton_ng` and `newtonstack_ng` are *unified*: normalized,
implicit and hybrid models are all handled by the same solver through a per-equation
residual mask, so there are no separate `_un_normalized` variants. The feedback-set solvers
`newton_fbmin_ng` and `newtonstack_fbmin_ng` are for normalized models only.

All common options above apply; the per-solver specifics follow.

### `xgenr_ng`, `sim_ng`, `sim1d_ng`, `res_ng`
Direct ports of their legacy counterparts (topological sweep, Gauss-Seidel 2-D / 1-D,
one-pass evaluation) on the shared skeleton — normalized models only, like the originals.
`res_ng` regenerates and byte-compares the transpiled jit file, so a stale compiled
evaluator is refreshed automatically.

### `newton_ng` — unified per-period Newton
Covers normalized **and** implicit equations in one solver, replacing the
`newton` / `newton_un_normalized` pair. The whole model is one system (every endogenous
variable is an unknown); there is no prolog/core/epilog split. One Jacobian is factorized at
the first period and carried forward; `nonlin` decides all refreshing.

Convergence is tested on the **residual**, equation by equation: implicit (`___RES`) rows
absolutely (`|G(y)| <= newton_absconv`), normalized rows relative to the variable's size
(`|F(y) - y| <= relconv * max(|y|, absconv)`).

| Option | Default | Meaning |
|---|---|---|
| `nonlin` | `False` | `N`: refresh the Jacobian every `N` iterations within a period; `-N`: also a fresh Jacobian before every period; `0`/`False`: one Jacobian for the whole span |
| `newton_absconv` | `0.001` | residual tolerance for implicit equations |
| `newton_reset`, `newtonalfa`, `newtonnodamp` | | as legacy `newton` |
| `lnjit` | `False` | jit the derivative model |
| `forcenum` | `False` | numerical derivatives |
| `diff_ng` | `0` | evaluate the derivative model through `res_ng` instead of legacy `res` |
| `permc_spec` | `None` (COLAMD) | SuperLU column ordering for the per-period LU |
| `factorizer` | `None` | `'umfpack'`: cvxopt/UMFPACK with the symbolic analysis cached and reused across periods and `nonlin` refreshes — pays off when nonlinear models refactorize often |

The former, more complex version (with a graph-based `split` option, backtracking and Gauss
rescues) is kept for comparison as `newton_ng_old` (`solver='newton_old'` in `solve_ng`).

### `newton_fbmin_ng` — per-period Newton on the minimal feedback set
Decomposes each period's core into a DAG part (solved exactly by topological sweeps) and a
*minimal feedback vertex set* that Newton iterates. Very fast when the feedback set is small —
on PAK (6 feedback variables) it is the fastest solver of all, with and without jit.

**Normalized models only:** the split comes from the model's dependency graph, which does
not see an implicit equation's dependence on its declared unknown.

| Option | Default | Meaning |
|---|---|---|
| `jacobian` | `'fd'` | `'fd'`: dense finite differences of the reduced residual; `'direct'`: `newton_diff` on the fb equations only (degenerates to fixed point when there are no direct fb-fb dependencies, which is the empirical norm); `'gauss'`: damped fixed point, no Jacobian (robustness fallback) |
| `nonlin` | `max(n_fb, 2)` | refresh interval (break-even is about the feedback set size) |

### `newtonstack_ng` — unified stacked Newton (alias `newtonstack_implicit_ng`)
The stacked solver for models with leads, normalized and implicit equations alike.
It follows legacy `newtonstack_implicit` step for step (identical residuals per iteration
on PAK).

| Option | Default | Meaning |
|---|---|---|
| `nonlin` | `False` | refresh the stacked Jacobian every `N` iterations. The scheduled refresh is **skipped once the residuals are within `newton_absconv`**: a rebuild costs many iterations (≈0.5 s on PAK against ≈0.05 s for the whole solve), and right at convergence it would never be used |
| `newton_absconv` | `0.001` | residual tolerance (max over all equations); convergence also requires the relative change of the unknowns to pass `relconv` |
| `newton_reset`, `newtonalfa`, `newtonnodamp` | | as legacy `newtonstack` |
| `nljit` | `0` | jit the derivative model (measured *slower* on FRB/US MCE — the derivative model is evaluated once per Jacobian build, so there is nothing for numba to amortize; leave off unless `nonlin` refreshes are frequent) |
| `nchunk` | `chunk` when jitted | derivative-model chunk size |
| `diff_ng` | `0` | evaluate the derivative model through `res_ng` |
| `permc_spec` | `'NATURAL'` | SuperLU ordering of the stacked LU; the band-preserving default is ~10x faster than COLAMD on block-banded stacked systems (13s to 1.2s on FRB/US MCE). Pass `'COLAMD'` to compare; never use `'MMD_AT_PLUS_A'` here (fill explosion) |
| `factorizer` | `None` | `'umfpack'`: symbolic analysis reused across rebuilds; measured a wash vs SuperLU/NATURAL on FRB/US (faster factorization, slower backsolves) |
| `split` | `0` | **stacked pro/core/epi split**: Newton (and the LU) only on variables involved in collapsed-graph cycles (variables depending on both their own lag *and* lead are promoted to core — a true stacked cycle); upstream *prolog* variables swept exactly once before, pure reporting *epilog* variables once after convergence. Handles implicit equations (`___RES` always core, dependencies on declared unknowns mapped to their residual equation). FRB/US MCE: 234 core / 29 prolog / 22 epilog, LU 1.23s to 0.71s |

The Jacobian build (derivative evaluation + sparse assembly + LU) is reported as a separate
`Jacobian build time` stats line — the total of all builds in the call, the one in `prepare`
and any `nonlin` refreshes — since the first runs outside the simulation-time measurement.

### `newtonstack_fbmin_ng` — stacked Newton on the stacked feedback set
The feedback-set idea applied to the *stacked* (period x variable) graph, so it also covers
leads. Non-feedback equations are solved exactly by topological sweeps that may run across
periods; Newton iterates only the stacked feedback unknowns. **Normalized models only**, like
`newton_fbmin_ng`.

Its niche: **lead-only recursive models** (empty feedback set — solved exactly by a single
stacked sweep, no Jacobian at all, which no other solver can do) and small models with small
stacked feedback sets. For large forward-looking models the reduced Jacobian's *dense*
`n_fb x n_fb` LU loses to `newtonstack_ng` + `split` — the decomposition-size print tells you
immediately which regime you are in. Inspect the feedback set via `mmodel.ng_stackfbmin_fb`.

The stacked sweep calls one small Python function per equation and period (30,048 calls per
sweep on PAK), so `ljit` does not speed it up: about 0.2 s on PAK with or without jit.

| Option | Default | Meaning |
|---|---|---|
| `jacobian` | `'stack'` | `'stack'`: Schur complement of the stacked Jacobian (exact stacked Newton step on the reduced system); `'fd'`: finite differences (`n_fb + 1` full-span sweeps per build — only for small fb sets); `'gauss'`: damped fixed point |
| `nonlin` | `False` (`'fd'`: `max(n_fb, 2)`) | refresh interval; as for `newtonstack_ng`, skipped once the feedback residuals are within `newton_absconv` |
| `newton_absconv` | `0.001` | feedback residual below which a scheduled refresh is skipped (convergence itself is the relative-change test on the feedback unknowns) |
| `keep_residual` | `False` | store the feedback residual as `mmodel.ng_stackfbmin_residual` |

`stats=True` reports the solver build time (`Solver build time`) next to the number of builds.

SCCs larger than `LARGE_SCC_COLLAPSE` (1000 nodes) use the feedback set of the collapsed
variable-level graph — sufficient and fast where the exact cycle-by-cycle heuristic would hang.

## Choosing a solver, and making it fast

1. **Trust the dispatch first.** The structural guess (`xgenr` / `sim` / `newton*` /
   `newtonstack*`) is correct for the model class; the `_ng` variants are drop-in
   replacements to opt into. Any implicit equation rules out the sweep and Gauss-Seidel
   solvers — Newton is mandatory.
2. **Hybrid models** (normalized and residual-form equations mixed) are dispatched to
   `newton_ng`, or `newtonstack_ng` with leads. On PAK in hybrid form `newton_ng` takes
   0.13 s (0.07 s with jit), the legacy alternatives about 25–30 s.
3. **Keep `use_fbmin=True`** (the default) for Gauss-Seidel: ordering the core by its
   feedback set made `sim` about 1.6 times faster on PAK.
4. **Try `newton_fbmin_ng`** on normalized backward-looking models with a small feedback set —
   on PAK it is the fastest solver, also ahead of jitted Gauss-Seidel.
5. **Jit the model** (`ljit=1`) when you solve more than a handful of times. The first call
   pays numba compilation (8–20 s on PAK); the transpiled file in `modelsource/` caches the
   compiled code across sessions. Warm solves are then 4–8 times faster.
6. **Reuse the Jacobian** in experiment loops with the `_ng` Newton solvers: `newton_reset=0`
   keeps the factorization from the previous solve, and Newton converges fine from a slightly
   stale Jacobian for moderate shocks (`nonlin` still refreshes mid-solve if a hard shock
   needs it). Legacy `newton` / `newton_un_normalized` must keep their default
   `newton_reset=1`.
7. **`split=1`** on `newtonstack_ng` removes the pure-reporting and pure-upstream variables
   from the stacked factorization at no cost in accuracy. It pays off without jit; with jit
   on PAK it was slower, and on a fully implicit model it gains nothing.
8. Leave the derivative model un-jitted (`nljit=0`) unless `nonlin` refreshes are frequent,
   and leave `chunk` at 30.

As a calibration point, the FRB/US model-consistent-expectations shock loop
(285 equations, 124 quarters, stacked system of ~35,000 unknowns):

| Configuration | Per solve |
|---|---|
| `newtonstack_ng`, COLAMD ordering, rebuild every solve | ~14.5 s |
| + `permc_spec='NATURAL'` (now the default) | ~3.0 s |
| + `split=1`, jit warm | ~2.1 s |
| + `newton_reset=0` (reuse factorization across shocks) | **~0.1-0.2 s** |

## Benchmark: all solvers on the PAK carbon-tax scenario

The Pakistan model (`pak.pcim`, no leads) solved for a carbon tax of 30 over 2020–2100
(81 years), in three equation forms, with every legacy and `_ng` solver that handles the
form. The test script is `modelflow/modeltest/speedtest.py`. Each solver runs twice:

* **cold** — code regenerated and compiled, Jacobians rebuilt (`transpile_reset=True`,
  `newton_reset=True`);
* **warm** — the same call again, with everything reused. This is the time that matters in
  an experiment loop. Exception: legacy `newton` and `newton_un_normalized` rebuild their
  Jacobian in the warm run too (their default, `newton_reset=1`), because they cannot reuse
  one — see below.

Options: `chunk=30`, `stringjit=True` (so every jit compile is genuinely fresh), `nonlin=6`
for the Newton solvers, `use_fbmin=True`. Times are wall-clock seconds for the whole model
call.

### Normalized model

| Solver | Warm, no jit | Warm, jit | Cold, jit (compile) |
|---|---:|---:|---:|
| `sim` | 0.27 | 0.036 | 12 |
| `sim_ng` | 0.28 | 0.041 | 12 |
| `sim1d` | 0.22 | 0.036 | 8 |
| `sim1d_ng` | 0.22 | 0.042 | 8 |
| `newton` | 4.1 | 4.1 | 13 |
| `newton_implicit` | 4.3 | 4.0 | 25 |
| `newton_ng` | 0.14 | 0.031 | 20 |
| `newton_fbmin_ng` | **0.10** | **0.025** | 11 |
| `newtonstack` | 0.19 – 0.75 | 0.05 – 0.6 | 13 |
| `newtonstack_implicit` | 0.19 | 0.047 | 14 |
| `newtonstack_ng` | 0.21 | 0.055 | 14 |
| `newtonstack_ng`, `split=1` | ≈ ½ of `newtonstack_ng` | 0.08 – 0.14 | |
| `newtonstack_fbmin_ng` | 0.21 | 0.21 | 4 |

### Implicit and hybrid models

The same model with every equation rewritten to residual form, `X___RES = (rhs) - (X)`
(*implicit*, via `modelmanipulation.un_normalize_model`), and with only the 75 estimated
equations rewritten (*hybrid*). Only Newton solvers can solve these. Warm times; this run
was about 1.5–2 times slower overall than the runs behind the normalized table, so compare
within this table:

| Solver | Implicit, no jit | Implicit, jit | Hybrid, no jit | Hybrid, jit |
|---|---:|---:|---:|---:|
| `newton_un_normalized` | 29 | 29 | | |
| `newton_implicit` | 28 | 30 | 24 | 30 |
| `newton_ng` | **0.20** | **0.049** | **0.13** | **0.073** |
| `newtonstack_un_normalized` | 0.33 | 0.065 | | |
| `newtonstack_implicit` | 0.29 | 0.086 | 0.22 | 0.091 |
| `newtonstack_ng` | 0.30 | 0.056 | 0.24 | 0.11 |
| `newtonstack_ng`, `split=1` | 0.32 | 0.54 | 0.22 | 0.27 |

* All of them reach the same solution as the Newton solvers on the normalized model.
* `newton_ng` is the clear choice for implicit and hybrid models without leads. The legacy
  per-period solvers take about 30 s: they rebuild a Jacobian of all 378 equations in every
  period, while `newton_ng` carries its Jacobian forward.
* `split=1` gains nothing on the implicit model — every residual equation is in the core —
  and is slower with jit, see below.

**Results.** All solvers agree with `sim_ng` to within about 1e-4 (relative). The Newton
solvers agree with each other much more closely; the difference to `sim_ng` is mostly where
Gauss-Seidel stops under its relative-change test (and that depends on the core order, see
`use_fbmin`). `sim` and `sim_ng` give identical results without jit.

**Jit** costs 8–20 s of compiling on the first call. What it buys depends on how much of a
solver's time goes into evaluating the model equations — the only thing jit compiles:

| Solver | Warm, no jit | Warm, jit | Speed-up |
|---|---:|---:|---:|
| `sim` | 0.27 | 0.036 | 7.5× |
| `sim_ng` | 0.28 | 0.041 | 6.8× |
| `sim1d` | 0.22 | 0.036 | 6.1× |
| `newton_ng` | 0.14 | 0.031 | 4.5× |
| `newton_fbmin_ng` | 0.10 | 0.025 | 4× |
| `newtonstack_ng` | 0.21 | 0.055 | 3.8× |
| `newtonstack_ng`, `split=1` | | | none |
| `newtonstack_fbmin_ng` | 0.21 | 0.21 | none |

* **Gauss-Seidel gains most.** It is almost nothing but equation evaluation — many sweeps
  per period over the whole core.
* **Newton gains less.** It needs only about 6–7 evaluations per period, and a larger share
  of its time goes to building and solving the sparse LU (scipy, already compiled) and to
  Python overhead per iteration. The stacked solvers have the largest such share.
* **The stacked per-equation evaluations gain nothing.** With `split=1` the core residual,
  the prolog and the epilog, and in `newtonstack_fbmin_ng` the stacked sweeps, are evaluated
  one equation at a time through Python functions that are not compiled.
* The legacy per-period solvers that rebuild their Jacobian every period (`newton`,
  `newton_implicit`, `newton_un_normalized`) gain nothing either: the rebuilds dominate.

Even so, the best Newton solvers stay ahead after jit: `newton_fbmin_ng` 0.025 s and
`newton_ng` 0.031 s against 0.036 s for `sim` — and without jit the gap is much larger
(0.10 s against 0.27 s).

**Legacy `newton`** solves PAK (1e-5 from `sim_ng`) **when it rebuilds its Jacobian on every
call**, which is its default (`newton_reset=1`). It cannot reuse a Jacobian across solves: it
keeps one solver, overwritten by every `nonlin` refresh with the Jacobian of the period being
solved, so after a solve it holds a late-period Jacobian. With `newton_reset=0` the next solve
starts 2020 from that Jacobian, diverges and fails in a `LOG` (a division by zero with jit).
`newton_ng` keeps a Jacobian per refreshed period and reuses them safely, which is what makes
it fast in experiment loops.

**Why legacy `newton` and `newton_implicit` take 4 s on the normalized model.** With
`nonlin=6`, every period needs 7 iterations: at iteration 6 the sum of changes is still just
above the tolerance (typically 0.03–2), so the Jacobian is rebuilt — in all 81 periods — and
one more iteration converges. `newton_ng` (0.14 s) needs no such rebuilds. The time is the
same with `use_fbmin` on or off.

**Fixes found by this benchmark (October 2026).**

* `newton_implicit` returned a wrong solution on normalized models: it iterated Newton on
  the simultaneous core but threw away the results of the recursive prolog and epilog. On a
  normalized model it now evaluates them in place around the core; on implicit or hybrid
  models it treats the whole model as one Newton system.
* `newtonstack_ng` with `split=1` returned a wrong solution on the hybrid model: identities
  lying between residual-form equations of the core were put in the epilog, so the core
  iterated against their stale values. Variables both downstream and upstream of the core
  now belong to the core. This also makes the normalized core somewhat larger, so `split`
  now roughly halves the time without jit (2.7 times faster before the fix).
* `newtonstack_ng` rebuilt its Jacobian on a scheduled `nonlin` iteration even when the
  solution had already converged; it now skips the rebuild once the residuals are within
  `newton_absconv`.
* Solving an implicit or hybrid model repeatedly with a frame that lacks the `X___RES`
  columns counted as new data on every call, so the solver code was regenerated — and with
  jit recompiled — every time. `model.is_newdata` now checks the column layout after the
  missing model variables are added.

**`split=1` on `newtonstack_ng`.** Newton and the stacked LU only cover the simultaneous
core; the prolog is swept once before, the epilog once after. Without jit this roughly
halves the time on the normalized model. With jit it is slower than without `split`,
because the core residual, prolog and epilog are evaluated equation by equation in Python.
On a fully implicit model the core is everything, so `split` gains nothing.

**Stacked Newton and the `nonlin` refresh.** The stacked solvers either take about 0.05 s or
about 0.6 s with jit. The difference is one Jacobian rebuild (≈0.5 s), scheduled by
`nonlin=6` at iteration 6. Legacy `newtonstack` tests a *sum* of changes against 0.001 and
lands just above or just below it at iteration 6 (0.0012 in some runs, 0.0009 in others),
so it sometimes pays for the rebuild. `newtonstack_ng` follows legacy
`newtonstack_implicit` step for step and runs as fast.

**`newtonstack_fbmin_ng`** does not use jit at all: its stacked sweeps call one small Python
function per equation and period. That is why its warm time is the same with and without
jit, and why its cold time (building the stacked graph and solver, no compiling) is short.

**Gauss-Seidel without `use_fbmin`** (normal block order): `sim` 0.44 s warm without jit,
0.059 s with jit — about 1.6 times slower than with the feedback ordering.

**How reliable the times are.** Measured on a laptop (Ryzen AI 9 HX PRO 370, ThinkPad P14s).
Its CPU mixes 4 fast and 8 slower cores and holds sustained load at about 28 W, so the same
run can be up to 2–3 times slower when Windows moves Python to a slow core or after a minute
or two of continuous compiling. A second source of outliers is Python's garbage collector:
late in a long session a collection pause of about 0.4 s can land in any solve. The speed test
therefore collects garbage before each timed call and switches the collector off while the
clock runs (`GC_QUIET = True`). The tables use runs and stretches without such slowdowns;
repeat a run before trusting a single slow number.

## Examples

A small model is enough to show the solver families at work. It has a simultaneous core
(`C` and `Y` depend on each other), a recursive prolog (`I` depends only on lags) and an
epilog (`T` is computed from the solution but feeds nothing back):

```
C = 0.6*Y + 0.2*C(-1) + 10        consumption
I = 0.3*(Y(-1) - Y(-2)) + 20      investment (accelerator, lags only)
Y = C + I + G                     income
T = 0.25*Y                        tax revenue
```

The examples solve a rise in government spending `G` from 50 to 60 from 2010 on, with
different solvers, and compare the solutions. Differences of the order of 1e-6 or smaller
are the convergence tolerance, not disagreement.

In [ ]:
import pandas as pd
from modelclass import model
import modelmanipulation as mp

small = '''
FRML <> C = 0.6*Y + 0.2*C(-1) + 10 $
FRML <> I = 0.3*(Y(-1) - Y(-2)) + 20 $
FRML <> Y = C + I + G $
FRML <> T = 0.25*Y $
'''
msmall = model(small, modelname='small')

years = range(2000, 2031)
df = pd.DataFrame({'G': 50.0, 'C': 100.0, 'I': 20.0, 'Y': 170.0, 'T': 40.0}, index=years)

base = msmall(df, 2002, 2030, solver='sim', silent=1, reset_options=True)
alt = base.copy()
alt.loc[2010:, 'G'] = 60.0

endo = ['C', 'I', 'Y', 'T']
print('prolog:', msmall.preorder, ' core:', msmall.coreorder, ' epilog:', msmall.epiorder)

### Normalized model: Gauss-Seidel and Newton

Every solver for normalized models reaches the same solution. `reset_options=True` on each
call keeps options from leaking between solvers (options are sticky). The table shows the
largest absolute difference from `sim_ng` per variable.

In [ ]:
solvers = ['sim', 'sim_ng', 'sim1d_ng', 'newton', 'newton_implicit', 'newton_ng', 'newton_fbmin_ng']
results = {s: msmall(alt, 2002, 2030, solver=s, silent=1, reset_options=True)
           for s in solvers}

ref = results['sim_ng'][endo]
pd.DataFrame({s: (r[endo] - ref).abs().max() for s, r in results.items()})

The effect of the spending rise itself, for reference:

In [ ]:
(results['sim_ng'][endo] - base[endo]).loc[2008:2015]

### Implicit and hybrid forms: why Newton is required

`modelmanipulation.un_normalize_model` rewrites equations to residual form,
`X___RES = (rhs) - (X)`. With `implicit=True` it rewrites every equation; without it, only the
equations tagged `IMPLICIT` in the FRML name — which gives a hybrid model. The model records
its form in `normalized` / `implicit` / `hybrid`.

No sweep can assign a `___RES` equation, so Gauss-Seidel cannot solve these. The fully implicit
model is solved by `newton_un_normalized` (the dispatcher's choice), and both forms by the
unified solvers `newton_ng` and `newton_implicit`. All reach the normalized solution. The
`___RES` columns that `alt` lacks are added (as 0) when solving.

In [ ]:
mimplicit = model(mp.un_normalize_model(msmall.equations, implicit=True),
                  modelname='small implicit')
# hybrid: only the consumption equation in residual form
mhybrid = model(mp.un_normalize_model(small.replace('FRML <> C =', 'FRML <IMPLICIT> C =')),
                modelname='small hybrid')

for m in (mimplicit, mhybrid):
    print(f'{m.name:15s} normalized={m.normalized} implicit={m.implicit} hybrid={m.hybrid}')
print(mhybrid.equations)

runs = [(mimplicit, 'newton_un_normalized'), (mimplicit, 'newton_implicit'),
        (mimplicit, 'newton_ng'),
        (mhybrid, 'newton_implicit'), (mhybrid, 'newton_ng')]
pd.DataFrame({(m.name, s): (m(alt, 2002, 2030, solver=s, silent=1, reset_options=True)[endo]
                            - ref).abs().max()
              for m, s in runs})

### A model with a lead: stacked Newton

Give consumption a forward-looking term, `C(+1)`. Now the periods cannot be solved one at a
time: every period depends on the next, and the whole span is one stacked system. The
per-period solvers are out; the stacked ones solve it. The data must extend past the last
solved period — here the 2030 values serve as the terminal condition for 2029.

`split=1` takes the epilog `T` out of the stacked Newton system. `I` stays in: it depends on
lagged `Y` and `Y` depends on `I`, which across periods is a cycle — in a stacked system
lags couple the periods just like leads. `newtonstack_fbmin_ng` iterates only on a feedback
set of the stacked graph.

In [ ]:
forward = small.replace('C = 0.6*Y + 0.2*C(-1) + 10', 'C = 0.5*Y + 0.3*C(+1) + 0.1*C(-1) + 10')
mforward = model(forward, modelname='small forward')
print('maxlead:', mforward.maxlead)

fbase = mforward(df, 2002, 2029, solver='newtonstack_ng', silent=1, reset_options=True)
falt = fbase.copy()
falt.loc[2010:, 'G'] = 60.0

stacked = [('newtonstack', {}), ('newtonstack_implicit', {}), ('newtonstack_ng', {}),
           ('newtonstack_ng', {'split': 1}), ('newtonstack_fbmin_ng', {})]
fres = {(s + (' split' if o else '')): mforward(falt, 2002, 2029, solver=s, silent=1,
                                                reset_options=True, **o)
        for s, o in stacked}
fref = fres['newtonstack_ng'][endo]
pd.DataFrame({s: (r[endo] - fref).abs().max() for s, r in fres.items()})